# PyTorch 入门第一课：张量操作（Tensor）与维度魔术

欢迎来到 PyTorch 的世界！在深度学习和大语言模型（LLM）中，**张量（Tensor）**是所有数据的基本单位。无论是文本、图像还是模型权重，它们在神经网络内部全部都表现为多维张量。

本节课我们将深入浅出地掌握以下核心概念：
1. **张量的创建与核心属性**（`shape`, `dtype`, `device`）
2. **维度魔术（核心难点）**（`view`/`reshape`, `squeeze`/`unsqueeze`, `permute`）
3. **矩阵运算**（逐元素乘法 `*` 与矩阵乘法 `@`）

## 1. 张量的创建与基本属性

我们先来学习如何创建张量，以及如何查看它的三大核心属性：
- `shape`：张量的形状（每个维度的大小，比如 `[2, 3]` 表示 2行3列）。
- `dtype`：张量的数据类型（如浮点数 `torch.float32`，整数 `torch.int64`）。
- `device`：张量所在的设备（CPU 或者是 Mac 的 GPU 加速芯片 `mps`）。

In [1]:
import torch

# 1.1 从已有的 Python 列表创建张量
a = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print("a:", a)
print("a 的形状 (shape):", a.shape)
print("a 的数据类型 (dtype):", a.dtype)
print("a 所在的设备 (device):", a.device)
print("-" * 50)

# 1.2 创建全零张量 (常用于初始化)
b = torch.zeros((2, 4))
print("b (zeros):\n", b)
print("-" * 50)

# 1.3 创建随机张量 (从正态分布中随机抽取，常用于初始化模型参数)
c = torch.randn((3, 2))
print("c (randn):\n", c)
print("-" * 50)

# 1.4 将张量移动到 Mac 的 GPU (MPS) 上加速
if torch.backends.mps.is_available():
    device = torch.device("mps")
    a_gpu = a.to(device)
    print("a_gpu 所在的设备:", a_gpu.device)
else:
    print("当前环境不支持 MPS 加速")

a: tensor([[1., 2., 3.],
        [4., 5., 6.]])
a 的形状 (shape): torch.Size([2, 3])
a 的数据类型 (dtype): torch.float32
a 所在的设备 (device): cpu
--------------------------------------------------
b (zeros):
 tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.]])
--------------------------------------------------
c (randn):
 tensor([[-0.4810,  0.4603],
        [-0.5442,  1.1120],
        [-1.1115, -2.0609]])
--------------------------------------------------
a_gpu 所在的设备: mps:0


## 2. 维度魔术（核心要点 🌟）

在处理大语言模型时，数据的形状经常变化（例如：`[batch_size, seq_len, hidden_dim]`）。你必须能随时改变它的形状。下面是 3 组最常用的维度操作方法：

### 2.1 改变形状：`view()` 与 `reshape()`
- 它们的作用是一样地，都是把张量拉伸或重塑。推荐优先使用 `view()`（效率极高，但要求张量在内存中连续），如果报错则用 `reshape()`。
- 技巧：可以使用 `-1` 让 PyTorch 自动推算某一个维度的长度。

In [2]:
x = torch.arange(12) # 创建一个 0 到 11 的一维张量
print("原始 x:", x, "形状:", x.shape)

# 变成 3 行 4 列
x_2d = x.view(3, 4)
print("\n重塑后的 x_2d:\n", x_2d, "形状:", x_2d.shape)

# 使用 -1 自动推算列数 (这里行数设为 2，列数会自动算成 6)
x_auto = x.view(2, -1)
print("\n自动计算列数的 x_auto:\n", x_auto, "形状:", x_auto.shape)

原始 x: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11]) 形状: torch.Size([12])

重塑后的 x_2d:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]]) 形状: torch.Size([3, 4])

自动计算列数的 x_auto:
 tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]]) 形状: torch.Size([2, 6])


### 2.2 增加/减少维度：`squeeze()` 与 `unsqueeze()`
- `unsqueeze(dim)`：在指定的 `dim` 位置**增加**一个大小为 1 的新维度。常用于给单个样本添加 `batch_size` 维度。
- `squeeze()`：**压缩**所有大小为 1 的维度。

In [3]:
y = torch.tensor([1, 2, 3])
print("y 形状:", y.shape) # 一维 [3]

# 在第 0 维插入一个大小为 1 的维度 -> 变成二维 [1, 3]
y_batch = y.unsqueeze(0)
print("y_batch 形状:", y_batch.shape)
print("y_batch 内容:", y_batch)

# 在第 1 维插入 -> 变成二维 [3, 1]
y_col = y.unsqueeze(1)
print("y_col 形状:", y_col.shape)

# 压缩掉所有为 1 的维度 -> 变回一维 [3]
y_squeezed = y_batch.squeeze()
print("y_squeezed 形状:", y_squeezed.shape)

y 形状: torch.Size([3])
y_batch 形状: torch.Size([1, 3])
y_batch 内容: tensor([[1, 2, 3]])
y_col 形状: torch.Size([3, 1])
y_squeezed 形状: torch.Size([3])


### 2.3 交换维度：`permute()`
- `permute(dims)`：按照指定的顺序重新排列维度的顺序。这在 Transformer 里的多头注意力机制（Multi-Head Attention）中至关重要，用来将形状从 `[Batch, SeqLen, Heads, HeadDim]` 调整为 `[Batch, Heads, SeqLen, HeadDim]` 方便矩阵乘法。

In [ ]:
# 模拟一个 Batch=2, 序列长度 SeqLen=3, 特征维度 HiddenDim=4 的张量
t = torch.randn(2, 3, 4)
print("原始 t 形状 [Batch, SeqLen, HiddenDim]:", t.shape)

# 将维度顺序修改为 [Batch, HiddenDim, SeqLen] (即把第 1 维和第 2 维交换)
# 这里的 (0, 2, 1) 代表原来的 0维放在第0，原来的 2维放在第1，原来的 1维放在第2
t_permuted = t.permute(0, 2, 1)
print("交换后的 t 形状:", t_permuted.shape)

## 3. 矩阵运算：点积/矩阵乘法 `@` 与逐元素相乘 `*` 的区别

- `*` (或 `torch.mul`)：**逐元素相乘（Element-wise Multiplication）**。两个张量形状必须完全相同（或满足广播机制），对应位置的元素相乘。
- `@` (或 `torch.matmul`)：**矩阵乘法（Matrix Multiplication）**。遵循线性代数的“左行乘右列”规则。比如形状 `[A, B]` 的矩阵只能和形状 `[B, C]` 的矩阵相乘，结果形状为 `[A, C]`。

In [4]:
m1 = torch.tensor([[1, 2],
                   [3, 4]])

m2 = torch.tensor([[2, 0],
                   [1, 2]])

# 3.1 逐元素乘法 *
res_mul = m1 * m2
print("逐元素乘法 * 结果:\n", res_mul)
# 解释: [[1*2, 2*0], [3*1, 4*2]] = [[2, 0], [3, 8]]
print("-" * 50)

# 3.2 矩阵乘法 @
res_matmul = m1 @ m2
print("矩阵乘法 @ 结果:\n", res_matmul)
# 解释: 
# 第一行第一列: 1*2 + 2*1 = 4
# 第一行第二列: 1*0 + 2*2 = 4
# 第二行第一列: 3*2 + 4*1 = 10
# 第二行第二列: 3*0 + 4*2 = 8

逐元素乘法 * 结果:
 tensor([[2, 0],
        [3, 8]])
--------------------------------------------------
矩阵乘法 @ 结果:
 tensor([[ 4,  4],
        [10,  8]])


## 4. 闯关挑战与动手练习 🏆

请修改下方代码，完成挑战：

In [ ]:
# 挑战 1：创建一个形状为 [3, 5] 的随机张量，将其数据类型打印出来，并将它移动到 MPS (GPU) 上。
# 请在此处写你的代码：


# 挑战 2：已知一个张量 shape 为 [4, 1, 8]，请通过操作将其形状转变为 [8, 4]。
# 提示：可以先用 squeeze() 去掉大小为 1 的维度，再用 permute() 交换维度；或者直接用 view() / reshape()。
t_test = torch.randn(4, 1, 8)
# 请在此处写你的代码：


